# ⚡ Zero-Shot Time Series Forecasting with Chronos-Bolt & SnoozeML

This notebook demonstrates how to serve **Amazon Chronos-Bolt**, a state-of-the-art pretrained time-series foundation model, as a custom, process-isolated Python worker model in SnoozeML.

> ⚠️ **Performance Trade-Off: Use Custom Workers as a Last Resort**
>
> Custom Python workers allow for the flexibility of running custom models and third-party libraries (such as `chronos-forecasting` or custom PyTorch code). However, they are **inherently slower** than SnoozeML's native compiled backends (ONNX Runtime, Candle Safetensors, and pure-Rust tree boosters), and **cold starts require significantly more time (in the seconds magnitude, typically 2s–10s+)** due to process creation and framework initialization.
>
> **Always prioritize native compiled formats first** (exporting to ONNX or using native Safetensors/Hessboost). Use custom Python workers **only as a last resort** when other native methods do not work or cannot be exported.

### What is Chronos-Bolt?
[**Chronos-Bolt**](https://github.com/amazon-science/chronos-forecasting) (by Amazon Science) uses patching and direct multi-step forecasting to generate zero-shot probabilistic quantile forecasts.

### Why SnoozeML's Custom Process Worker?
Chronos-Bolt relies on PyTorch and the custom `chronos-forecasting` library. Serving it through SnoozeML's `worker-backend` provides:
1. **Scale-to-Zero VRAM/RAM Reclamation**: Because PyTorch CUDA context allocators hold onto GPU memory, terminating the child process upon inactivity returns **100% of memory to the OS**.
2. **Zero GIL Contention**: Multi-replica autoscaling launches independent processes across GPU streams and CPU cores without Python GIL locking.
3. **Dynamic Micro-Batching**: SnoozeML coalesces concurrent time series requests along `Axis(0)` without changing the model code.

In [12]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from snoozeml import SnoozeClient

print("SnoozeML client ready.")

SnoozeML client ready.


## 1. Inspect the Custom Worker Model Files

A custom worker model in SnoozeML consists of a directory with:
- `snooze_model.py`: Duck-typed class exposing `load(model_dir, device)` and `predict(inputs)`.
- `snooze_model.json`: Optional manifest configuring class name, process timeouts, and environment variables.

In [13]:
model_dir = Path("./chronos_bolt")
if not model_dir.exists():
    model_dir = Path("../models/chronos_bolt")

# Display the manifest
manifest_path = model_dir / "snooze_model.json"
with open(manifest_path, "r", encoding="utf-8") as f:
    print("--- snooze_model.json ---")
    print(f.read())

# Display snippet of the entrypoint
entrypoint_path = model_dir / "snooze_model.py"
with open(entrypoint_path, "r", encoding="utf-8") as f:
    code = f.read()
    print("--- snooze_model.py (first 40 lines) ---")
    print("\n".join(code.splitlines()[:40]))

--- snooze_model.json ---
{
  "entrypoint": "snooze_model.py",
  "class": "Model",
  "load_timeout_secs": 600,
  "request_timeout_secs": 120,
  "tensor_format": "numpy",
  "env": {
    "TOKENIZERS_PARALLELISM": "false",
    "OMP_NUM_THREADS": "1"
  }
}

--- snooze_model.py (first 40 lines) ---
"""Chronos-Bolt Time Series Foundation Model Worker for SnoozeML.

This module provides a duck-typed SnoozeML custom model for Amazon Science's
Chronos-Bolt zero-shot probabilistic time series forecasting models
(e.g., ``amazon/chronos-bolt-mini`` or ``amazon/chronos-bolt-small``).

It loads the Chronos-Bolt pipeline using the ``chronos-forecasting`` library,
handles device placement, and computes multi-step quantile forecasts across
dynamically batched time series inputs along Axis(0).
"""

from __future__ import annotations

import os
from typing import Any, Dict


class Model:
    """SnoozeML worker model class for Chronos-Bolt forecasting."""

    def __init__(self) -> None:
        self.pipe

## 2. Local Smoke Testing with `--check` (No Daemon Required)

Before starting or configuring the SnoozeML daemon, you can verify your custom model directly using the embedded worker runner:

```bash
# Run smoke test on CPU with a mock 1x32 input context
python -m snoozeml.worker --check examples/chronos_bolt --input context:1x32 --device cpu
```

The runner honors `snooze_model.json`, sets up isolated streams, executes `load()`, performs a dummy forward pass, and prints output shapes and sample values.

## 3. Configure and Connect to the SnoozeML Daemon

In `models.toml`, register the model for automated scale-to-zero and micro-batching:

```toml
[[models]]
id = "chronos_bolt"
path = "examples/chronos_bolt"
device = "cpu"                  # Use "cuda" for GPU acceleration
max_batch_size = 8             # Coalesce up to 8 time-series requests
max_batch_delay_ms = 15        # 15ms batch accumulation window
idle_timeout_secs = 120        # Scale-to-zero memory unload after 2 min
```

Connect with `SnoozeClient`:

In [14]:
client = SnoozeClient(endpoint="http://localhost:8080")

# Check server health
assert client.health_live(), "SnoozeML daemon is not reachable!"
print("Connected to SnoozeML server at http://localhost:8080")

# Query registered model status
model_info = client.models.get("chronos_bolt")
print(f"Model: {model_info.id} | State: {model_info.state.value} | Pinned: {model_info.pinned}")

Connected to SnoozeML server at http://localhost:8080
Model: chronos_bolt | State: idle | Pinned: False


## 4. Prepare Time Series Context Data

We prepare historical time series observations. Here we create an monthly trend with seasonality (similar to classic transportation or sales series).

In [15]:
# Generate 36 months of historical time series data (trend + seasonality + noise)
np.random.seed(42)
time_steps = np.arange(36, dtype=np.float32)
trend = 100.0 + 2.5 * time_steps
seasonality = 15.0 * np.sin(2 * np.pi * time_steps / 12)
noise = np.random.normal(0, 2.0, size=len(time_steps)).astype(np.float32)

series = trend + seasonality + noise

# Format as 2D batch tensor of shape (batch_size=1, context_length=36)
context_input = series.reshape(1, -1).astype(np.float32)
print(f"Input context shape: {context_input.shape}")
print(f"First 6 context values: {context_input[0, :6]}")

Input context shape: (1, 36)
First 6 context values: [100.99343 109.72347 119.28576 125.54606 122.52207 119.53172]


## 5. Cold Start vs. Warm In-Memory Inference

1. **Cold Start**: When the model is in `idle` state, the first request launches the worker process, initializes PyTorch, and loads model weights on-demand. **Cold starts take seconds (e.g. 2s–10s+)** due to Python process creation and model loading.
2. **Warm Execution**: Subsequent requests reuse the resident worker process over binary streams for fast in-memory execution.

In [16]:
# Ensure model starts in idle state (scale-to-zero demonstration)
client.models.evict("chronos_bolt")
print(f"Initial State: {client.models.get('chronos_bolt').state.value}")

# --- 1st Request: Cold Start (spawns worker process + loads model) ---
resp_cold = client.infer("chronos_bolt", inputs={"context": context_input})
print(f"❄️  Cold Start Latency: {resp_cold.duration_ms:.2f} ms (process spawn + weight load)")
print(f"State after cold start: {client.models.get('chronos_bolt').state.value}")

# --- 2nd Request: Warm Execution (in-memory binary pipe execution) ---
resp_warm = client.infer("chronos_bolt", inputs={"context": context_input})
print(f"🔥 Warm Latency:        {resp_warm.duration_ms:.2f} ms (resident in-memory inference)")

speedup = resp_cold.duration_ms / max(resp_warm.duration_ms, 0.01)
print(f"⚡ Speedup:             {speedup:.1f}x faster warm execution")

Initial State: idle
❄️  Cold Start Latency: 4504.93 ms (process spawn + weight load)
State after cold start: ready
🔥 Warm Latency:        24.63 ms (resident in-memory inference)
⚡ Speedup:             182.9x faster warm execution


## 6. Inspect Probabilistic Forecasts

Chronos-Bolt produces:
- `forecast`: shape `(1, 9, 12)` covering 9 quantile levels `[0.1, 0.2, ..., 0.9]` over the 12-step horizon.
- `median`: shape `(1, 12)` representing the 50th percentile point forecast.

In [18]:
quantiles = resp_warm.outputs["forecast"].as_numpy()[0]
median_pred = resp_warm.outputs["median"].as_numpy()[0]

# Extract 10th percentile, median (50th), and 90th percentile
if quantiles.ndim == 2 and quantiles.shape[0] == 9:
    p10, p50, p90 = quantiles[0, :], quantiles[4, :], quantiles[8, :]
elif quantiles.ndim == 2 and quantiles.shape[1] == 9:
    p10, p50, p90 = quantiles[:, 0], quantiles[:, 4], quantiles[:, 8]
else:
    p50 = median_pred
    p10 = median_pred * 0.9
    p90 = median_pred * 1.1

forecast_df = pd.DataFrame({
    "Step": [f"t+{i+1}" for i in range(len(p50))],
    "10th Percentile": np.round(p10, 2),
    "Median (50th)": np.round(p50, 2),
    "90th Percentile": np.round(p90, 2),
})
print("12-Step Ahead Forecast Summary:")
print(forecast_df.to_string(index=False))

12-Step Ahead Forecast Summary:
Step  10th Percentile  Median (50th)  90th Percentile
 t+1       171.880005     181.119995       190.139999
 t+2       171.740005     185.929993       199.360001
 t+3       171.880005     190.600006       208.119995
 t+4       170.199997     192.740005       214.179993
 t+5       166.080002     191.949997       216.779999
 t+6       161.619995     190.490005       218.949997
 t+7       155.860001     187.570007       218.839996
 t+8       150.080002     184.460007       218.089996
 t+9       145.470001     181.910004       217.899994
t+10       142.789993     181.699997       220.000000
t+11       141.419998     182.789993       222.660004
t+12       140.839996     184.690002       226.169998


## 7. Scale-to-Zero Verification

When no requests arrive within `idle_timeout_secs` (or when evicted explicitly via API), SnoozeML shuts down the worker process and reaps it. This completely frees GPU VRAM and system RAM without memory leaks or residual CUDA allocations.

In [ ]:
# Evict the model
client.models.evict("chronos_bolt")
status = client.models.get("chronos_bolt")
print(f"Model state after scale-to-zero / evict: {status.state.value}")
assert status.state.value == "idle"
print("✅ Scale-to-zero verified: process terminated, memory fully reclaimed.")